In [ ]:
import os
import zipfile
import urllib.request
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# 1. Define secure paths and download the official NIH dataset zip file
dataset_url = "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"
zip_path = "cell_images.zip"
extract_path = "malaria_data"

if not os.path.exists(extract_path):
    print("Streaming 330MB NIH Malaria Dataset... (This may take 30-45 seconds)")
    urllib.request.urlretrieve(dataset_url, zip_path)

    print("Unpacking clinical image matrix...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    os.remove(zip_path) # Clean up space
    print("Unpacking complete!")
else:
    print("Dataset already active in workspace.")

# 2. Point to the unpacked root folder
data_dir = os.path.join(extract_path, "cell_images")
print(f"Verified internal folders: {os.listdir(data_dir)}")

Streaming 330MB NIH Malaria Dataset... (This may take 30-45 seconds)
Unpacking clinical image matrix...
Unpacking complete!
Verified internal folders: ['Parasitized', 'Uninfected']


In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

data_dir = 'malaria_data/cell_images'

# 1. Training transforms include RANDOM FLIPS for orientation invariance
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Validation transforms stay completely objective (No flips)
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Custom wrapper to cleanly separate train/val augmentations after splitting
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform
    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        return self.transform(img), label
    def __len__(self):
        return len(self.base_dataset)

# 3. Load the raw clinical images from disk
raw_dataset = ImageFolder(root=data_dir)
print(f"Identified Medical Classes: {raw_dataset.classes}")

# 4. Execute a clean 80% Train / 20% Validation split
train_size = int(0.8 * len(raw_dataset))
val_size = len(raw_dataset) - train_size
train_base, val_base = random_split(raw_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_dataset = AugmentedDataset(train_base, train_transform)
val_dataset = AugmentedDataset(val_base, val_transform)

# 5. Build high-speed DataLoaders for your T4 GPU
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print("\n--- Clinical Vision Pipeline Configured ---")
print(f"Total Unique Cells Audited: {len(raw_dataset)}")
print(f"Training Batch Count: {len(train_loader)}")
print(f"Validation Batch Count: {len(val_loader)}")

Identified Medical Classes: ['Parasitized', 'Uninfected']

--- Clinical Vision Pipeline Configured ---
Total Unique Cells Audited: 27558
Training Batch Count: 345
Validation Batch Count: 87


In [ ]:
import torch.nn as nn

class MalariaCNN(nn.Module):
    def __init__(self):
        super(MalariaCNN, self).__init__()

        # Block 1: Input (3, 64, 64) -> Output (16, 32, 32)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: Input (16, 32, 32) -> Output (32, 16, 16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Block 3: Input (32, 16, 16) -> Output (64, 8, 8)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

        # Fully Connected Diagnostic Classifier Head
        # 64 channels * 8x8 spatial grid size = 4096 inputs
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 2)  # 2 output classes: Parasitized vs Uninfected

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2) # Prevents overfitting on specific cell markings

    def forward(self, x):
        # Feature extraction phase
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))

        # Flattening the 3D tensor maps down to a 1D vector
        x = x.view(-1, 64 * 8 * 8)

        # Final fully connected inference layers
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Initialize the model and cast its arrays onto the GPU memory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_malaria = MalariaCNN().to(device)

print("--- Clinical CNN Architecture Successfully Compiled ---")
print(model_malaria)

--- Clinical CNN Architecture Successfully Compiled ---
MalariaCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_malaria.parameters(), lr=0.001)
epochs = 5

print("--- Initializing Malaria Diagnostic Training Loop ---")
for epoch in range(epochs):
    # --- TRAINING TRACKING ---
    model_malaria.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_malaria(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct_train / total_train) * 100

    # --- VALIDATION TRACKING ---
    model_malaria.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_malaria(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = (val_correct / val_total) * 100

    print(f"Epoch [{epoch+1}/{epochs}] -> Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("\n--- Training Sequence Finalized ---")

--- Initializing Malaria Diagnostic Training Loop ---
